# SNR-Guided Internal Reference Transfer for LumaGuard-Net
## Lightweight Self-Supervised Low-Light Image Enhancement on Kaggle GPU

> **Objective**: Exploit high-SNR reliable regions of a low-light image as internal structural references to guide the enhancement of low-SNR regions through confidence-gated feature transfer.

**Hardware Recommendation**: Kaggle **GPU T4** or **P100** (Turn on GPU in Kaggle Notebook Settings: `Accelerator -> GPU T4 x2` or `P100`).

In [ ]:
# 1. Verify GPU Environment & PyTorch CUDA Acceleration
import torch
import os

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: GPU is not enabled! Please go to Notebook Settings -> Accelerator -> GPU T4 or P100.")

!nvidia-smi

In [ ]:
# 2. Locate or Download Dataset
from pathlib import Path

# Check for pre-existing dataset in /kaggle/input or local repo
dataset_found = False
search_roots = [Path("/kaggle/input"), Path("data"), Path(".")]
train_path = None

for root in search_roots:
    if root.exists():
        for p in root.rglob("our485/low"):
            if p.is_dir() and len(list(p.glob("*.png"))) > 0:
                train_path = p.parent.parent
                dataset_found = True
                print(f"Found LOL Dataset at: {train_path}")
                break
    if dataset_found:
        break

if not dataset_found:
    print("Downloading LOL dataset...")
    !mkdir -p data
    !wget -q https://github.com/flyywh/CVPR-2020-Semi-Low-Light/raw/master/data/eval15.zip -O data/eval15.zip
    !unzip -q data/eval15.zip -d data/lol_dataset/
    print("Dataset prepared in data/lol_dataset/")

In [ ]:
# 3. Run Self-Supervised Training on GPU
# Automatic Mixed Precision (AMP) enables fast convergence (~5-10 seconds per epoch)
!python scripts/train_kaggle.py \
    --config configs/kaggle_gpu.yaml \
    --epochs 35 \
    --batch_size 16

In [ ]:
# 4. Evaluate Checkpoint on LOL-v1 eval15 Paired Benchmark
import glob

# Find best checkpoint
ckpt_candidates = glob.glob("/kaggle/working/**/best_checkpoint.pth", recursive=True) + glob.glob("experiments/**/best_checkpoint.pth", recursive=True)
best_ckpt = ckpt_candidates[0] if ckpt_candidates else "experiments/snr_reference_kaggle_gpu/best_checkpoint.pth"
print(f"Evaluating Checkpoint: {best_ckpt}")

!python scripts/evaluate.py \
    --checkpoint {best_ckpt} \
    --low_dir data/lol_dataset/eval15/low \
    --high_dir data/lol_dataset/eval15/high \
    --output_csv /kaggle/working/eval15_metrics.csv \
    --device cuda

In [ ]:
# 5. Run Full Diagnostic Inference on Test Images
!python scripts/infer.py \
    --input data/lol_dataset/eval15/low \
    --checkpoint {best_ckpt} \
    --output /kaggle/working/outputs/snr_enhanced \
    --diagnostics \
    --device cuda

In [ ]:
# 6. Visualize Enhanced Results Inline
import matplotlib.pyplot as plt
from PIL import Image

sample_comps = list(Path("/kaggle/working/outputs/snr_enhanced").glob("*_comparison.png"))
if not sample_comps:
    sample_comps = list(Path("outputs/snr_enhanced").glob("*_comparison.png"))

for comp_path in sample_comps[:4]:
    img = Image.open(comp_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.title(f"Left: Original Low-Light | Right: SNR-Guided Enhanced ({comp_path.stem})")
    plt.axis("off")
    plt.show()